In [ ]:
import os
import json
from pathlib import Path
import torch
import pandas as pd
import torch.nn.functional as F
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from tqdm import tqdm

output_csv = "clip_scores_ground_truth.csv"
jsonl_path = Path("../../data-transformation/ui_validation_captions.jsonl")
img_dir = Path("../validation_samples")
model_id = "openai/clip-vit-base-patch32"

def calculate_clip_score(model, processor, image, text, device):
    """
    Calcula a similaridade de cosseno entre os embeddings de imagem e texto.
    Multiplicamos por 100 para seguir o padrão de legibilidade da métrica CLIP Score.
    """
    inputs = processor(text=[text], images=image, return_tensors="pt", padding=True, truncation=True).to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
        
    image_embeds = outputs.image_embeds
    text_embeds = outputs.text_embeds
    
    # Normalizar os embeddings para calcular a similaridade de cosseno
    image_embeds = F.normalize(image_embeds, p=2, dim=-1)
    text_embeds = F.normalize(text_embeds, p=2, dim=-1)
    
    # Produto escalar entre os vetores normalizados
    cos_sim = torch.sum(image_embeds * text_embeds, dim=-1).item()
    
    # Escalar para um valor entre 0 e 100 (evitando valores negativos)
    clip_score = max(0, cos_sim) * 100 
    return round(clip_score, 4)

def save_df(results):
    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False, encoding='utf-8')
    
    # Exibir métricas agregadas
    mean_score = df["clip_score"].mean()
    
    print("\n" + "="*40)
    print("AVALIAÇÃO CONCLUÍDA")
    print("="*40)
    print(f"Total de imagens processadas: {len(df)}")
    print(f"CLIP Score Médio: {mean_score:.2f}")
    print(f"Resultados salvos com sucesso em: {output_csv}")
    
    # Exibir os top 3 piores e melhores scores para rápida validação
    print("\nTop 3 Piores Scores (Verificar alinhamento do domínio):")
    print(df.nsmallest(3, "clip_score")[["filename", "clip_score"]].to_string(index=False))
    
    print("\nTop 3 Melhores Scores:")
    print(df.nlargest(3, "clip_score")[["filename", "clip_score"]].to_string(index=False))

/home/matheus_mendes/miniconda3/envs/lora/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
model = CLIPModel.from_pretrained(model_id).to("cuda")
processor = CLIPProcessor.from_pretrained(model_id)

valid_samples = []
with open(jsonl_path, 'r', encoding='utf-8') as f:
    for line in f:
        if not line.strip():
            continue
            
        data = json.loads(line.strip())
        caption = data.get("caption", "").strip()
            
        img_path = os.path.join(img_dir, data.get("filename", ""))
        
        # Verificar se a imagem realmente existe na pasta
        if not os.path.exists(img_path):
            print(f"Aviso: Imagem ignorada pois não foi encontrada - {img_path}")
            continue
            
        valid_samples.append({
            "filename": data["filename"],
            "caption": caption,
            "filepath": img_path
        })
        
print(f"Total de amostras válidas para processar: {len(valid_samples)}")

# Calcular o CLIP Score para cada amostra
results = []
    
for sample in tqdm(valid_samples, desc="Calculando CLIP Scores"):
    try:
        # Carregar a imagem e converter para RGB (evita erros com imagens RGBA/escala de cinza)
        image = Image.open(sample["filepath"]).convert("RGB")
        
        score = calculate_clip_score(model, processor, image, sample["caption"], "cuda")
        
        results.append({
            "filename": sample["filename"],
            "caption": sample["caption"],
            "clip_score": score
        })
    except Exception as e:
        print(f"\nErro ao processar {sample['filename']}: {e}")

if results:
    save_df(results)
else:
    print("\nNenhuma amostra foi processada com sucesso.")

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 623.89it/s, Materializing param=visual_projection.weight]                                
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Aviso: Imagem ignorada pois não foi encontrada - ../validation_samples/2579_output.png
Total de amostras válidas para processar: 42


Calculando CLIP Scores: 100%|██████████| 42/42 [00:02<00:00, 17.88it/s]


AVALIAÇÃO CONCLUÍDA
Total de imagens processadas: 42
CLIP Score Médio: 33.60
Resultados salvos com sucesso em: clip_scores_ground_truth.csv

Top 3 Piores Scores (Verificar alinhamento do domínio):
        filename  clip_score
34216_output.png     25.3116
33178_output.png     25.4698
 3458_output.png     26.2277

Top 3 Melhores Scores:
            filename  clip_score
Rico_2349_output.png     46.3769
     3457_output.png     44.0942
    33576_output.png     41.6578
